# Satellite Orbit Propagation API Client

This notebook:
1. Reads satellite state vectors from an input JSON file
2. Calls the PINN API endpoint to get trajectory predictions
3. Saves API responses to an output JSON file
4. Generates 3D trajectory plots and saves them as images

In [14]:
# Cell 1: Imports & Configuration
import json
import requests
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Configuration - modify these paths as needed
API_URL = "https://dev-pinn.bosonqpsi.com/pinn"
INPUT_FILE = "input_states.json"      # Path to input file with satellite states
OUTPUT_FILE = "api_responses.json"    # Path to save API responses
PLOT_OUTPUT = "trajectory_plot.png"   # Path to save trajectory plot

In [15]:
# Cell 2: Load Input File Function
def load_satellite_states(filepath: str) -> list:
    """
    Parse input file containing satellite states.
    Supports both JSON array format [ {...}, {...} ] and concatenated JSON objects.
    
    Args:
        filepath: Path to the input JSON file
        
    Returns:
        List of satellite state dictionaries
    """
    with open(filepath, 'r') as f:
        content = f.read().strip()
    
    # Try to parse as JSON array first
    try:
        parsed = json.loads(content)
        if isinstance(parsed, list):
            print(f"Loaded {len(parsed)} satellite state(s) from {filepath} (JSON array format)")
            return parsed
        elif isinstance(parsed, dict):
            # Single object, return as list with one item
            print(f"Loaded 1 satellite state from {filepath} (single JSON object)")
            return [parsed]
    except json.JSONDecodeError:
        # Not a valid JSON array/object, try parsing as concatenated objects
        pass
    
    # Parse multiple concatenated JSON objects
    decoder = json.JSONDecoder()
    states = []
    idx = 0
    
    while idx < len(content):
        # Skip whitespace
        while idx < len(content) and content[idx] in ' \t\n\r':
            idx += 1
        if idx >= len(content):
            break
            
        try:
            obj, end_idx = decoder.raw_decode(content[idx:])
            states.append(obj)
            idx += end_idx
        except json.JSONDecodeError as e:
            print(f"Error parsing JSON at position {idx}: {e}")
            break
    
    print(f"Loaded {len(states)} satellite state(s) from {filepath} (concatenated format)")
    return states

In [16]:
# Cell 3: API Call Function
def call_pinn_api(payload: dict) -> dict:
    """
    Call the PINN API endpoint to get trajectory prediction.
    
    Args:
        payload: Dictionary with initial_position, initial_velocity, 
                 T_STEP_DURATION, N_STEPS, POINTS_PER_STEP, start_date
                 
    Returns:
        API response dictionary with 'trajectory' key, or error info
    """
    headers = {
        'Content-Type': 'application/json',
        'accept': 'application/json'
    }
    
    try:
        response = requests.post(API_URL, json=payload, headers=headers, timeout=60)
        
        if response.status_code == 200:
            return {'success': True, 'data': response.json()}
        elif response.status_code == 422:
            return {'success': False, 'error': 'Validation error', 'details': response.json()}
        else:
            return {'success': False, 'error': f'HTTP {response.status_code}', 'details': response.text}
            
    except requests.exceptions.Timeout:
        return {'success': False, 'error': 'Request timeout'}
    except requests.exceptions.ConnectionError as e:
        return {'success': False, 'error': f'Connection error: {e}'}
    except Exception as e:
        return {'success': False, 'error': f'Unexpected error: {e}'}

In [17]:
# Cell 4: 3D Plotting Function
def plot_trajectory_3d(trajectory: list, title: str = "Satellite Trajectory", 
                       save_path: str = None, color: str = 'b', label: str = None):
    """
    Generate a 3D plot from trajectory points.
    
    Args:
        trajectory: List of [x, y, z] coordinates in meters
        title: Plot title
        save_path: Path to save PNG (optional)
        color: Line color
        label: Legend label
        
    Returns:
        matplotlib Figure object
    """
    traj = np.array(trajectory)
    
    # Scale to kilometers for readability
    traj_km = traj / 1000.0
    
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot trajectory line
    ax.plot(traj_km[:, 0], traj_km[:, 1], traj_km[:, 2], 
            color=color, linewidth=1.5, label=label or 'Trajectory')
    
    # Mark start and end points
    ax.scatter(traj_km[0, 0], traj_km[0, 1], traj_km[0, 2], 
               color='green', s=100, marker='o', label='Start')
    ax.scatter(traj_km[-1, 0], traj_km[-1, 1], traj_km[-1, 2], 
               color='red', s=100, marker='s', label='End')
    
    ax.set_xlabel('X [km]')
    ax.set_ylabel('Y [km]')
    ax.set_zlabel('Z [km]')
    ax.set_title(title)
    ax.legend()
    ax.grid(True)
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Plot saved to {save_path}")
    
    return fig


def plot_multiple_trajectories(results: list, title: str = "Satellite Trajectories",
                               save_path: str = None):
    """
    Plot multiple trajectories on a single 3D figure.
    
    Args:
        results: List of result dictionaries with 'response' containing trajectory
        title: Plot title
        save_path: Path to save PNG (optional)
        
    Returns:
        matplotlib Figure object
    """
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(results)))
    
    for i, result in enumerate(results):
        if not result.get('success', False):
            continue
            
        trajectory = result['response'].get('trajectory', [])
        if not trajectory:
            continue
            
        traj = np.array(trajectory) / 1000.0  # Convert to km
        
        ax.plot(traj[:, 0], traj[:, 1], traj[:, 2], 
                color=colors[i], linewidth=1.5, label=f'Satellite {i+1}')
        
        # Mark start point
        ax.scatter(traj[0, 0], traj[0, 1], traj[0, 2], 
                   color=colors[i], s=50, marker='o')
    
    ax.set_xlabel('X [km]')
    ax.set_ylabel('Y [km]')
    ax.set_zlabel('Z [km]')
    ax.set_title(title)
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True)
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Plot saved to {save_path}")
    
    return fig

In [18]:
# Cell 5: Process All Satellites & Save Results
def process_satellites(input_file: str, output_file: str) -> list:
    """
    Load satellite states, call API for each, and save all responses.
    
    Args:
        input_file: Path to input JSON file with satellite states
        output_file: Path to save API responses
        
    Returns:
        List of result dictionaries
    """
    # Load satellite states
    states = load_satellite_states(input_file)
    
    if not states:
        print("No satellite states found in input file.")
        return []
    
    results = []
    
    for i, state in enumerate(states):
        print(f"\nProcessing satellite {i+1}/{len(states)}...")
        
        # Call API
        api_result = call_pinn_api(state)
        
        # Build result entry
        result = {
            'satellite_index': i + 1,
            'input': state,
            'success': api_result['success']
        }
        
        if api_result['success']:
            result['response'] = api_result['data']
            num_points = len(api_result['data'].get('trajectory', []))
            print(f"  Success: Received {num_points} trajectory points")
        else:
            result['error'] = api_result.get('error', 'Unknown error')
            result['details'] = api_result.get('details')
            print(f"  Failed: {result['error']}")
        
        results.append(result)
    
    # Save results to output file
    with open(output_file, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {output_file}")
    
    # Print summary
    success_count = sum(1 for r in results if r['success'])
    print(f"\nSummary: {success_count}/{len(results)} successful API calls")
    
    return results

In [19]:
# Cell 6: Main Execution

# Process all satellites and save responses
results = process_satellites(INPUT_FILE, OUTPUT_FILE)

# Generate and save trajectory plots
if results:
    successful_results = [r for r in results if r.get('success', False)]
    
    if successful_results:
        print(f"\nGenerating {len(successful_results)} individual trajectory plot(s)...")
        
        plot_files = []
        colors = plt.cm.tab10(np.linspace(0, 1, len(successful_results)))
        
        # Generate individual plot for each satellite
        for i, result in enumerate(successful_results):
            satellite_idx = result.get('satellite_index', i + 1)
            trajectory = result['response'].get('trajectory', [])
            
            if trajectory:
                # Generate unique filename for each plot
                plot_filename = f"trajectory_plot_satellite_{satellite_idx}.png"
                plot_files.append(plot_filename)
                
                # Create individual plot
                fig = plot_trajectory_3d(
                    trajectory,
                    title=f"Satellite {satellite_idx} Trajectory",
                    save_path=plot_filename,
                    color=colors[i],
                    label=f'Satellite {satellite_idx}'
                )
                plt.close(fig)  # Close to free memory
        
        print(f"\nDone! Output files:")
        print(f"  - API responses: {OUTPUT_FILE}")
        for plot_file in plot_files:
            print(f"  - Trajectory plot: {plot_file}")
    else:
        print("No successful API responses to plot.")
else:
    print("No results to process.")

Loaded 5 satellite state(s) from input_states.json (JSON array format)

Processing satellite 1/5...
  Success: Received 50 trajectory points

Processing satellite 2/5...
  Success: Received 50 trajectory points

Processing satellite 3/5...
  Success: Received 50 trajectory points

Processing satellite 4/5...
  Success: Received 50 trajectory points

Processing satellite 5/5...
  Success: Received 50 trajectory points

Results saved to api_responses.json

Summary: 5/5 successful API calls

Generating 5 individual trajectory plot(s)...
Plot saved to trajectory_plot_satellite_1.png
Plot saved to trajectory_plot_satellite_2.png
Plot saved to trajectory_plot_satellite_3.png
Plot saved to trajectory_plot_satellite_4.png
Plot saved to trajectory_plot_satellite_5.png

Done! Output files:
  - API responses: api_responses.json
  - Trajectory plot: trajectory_plot_satellite_1.png
  - Trajectory plot: trajectory_plot_satellite_2.png
  - Trajectory plot: trajectory_plot_satellite_3.png
  - Trajecto